# Task 1 — Preparazione del dataset (Bank Marketing)

**Corso:** Fondamenti e Applicazioni del Machine Learning (FML 2026)

In questo notebook trasformiamo il dataset grezzo in un formato comprensibile dal
classificatore ed estraiamo due file:

| File | Dimensione | Scopo |
|------|-----------|-------|
| `manuale.csv` | 12 istanze | calcoli **a mano** dei classificatori (Task 2) |
| `training.csv` | dataset completo pulito | EDA e addestramento (Task 3, 4, 5) |

**Riferimenti del corso:**
- Libro di testo: Witten, Frank, Hall, Pal — *Data Mining* (4ª ed.), cap. 4
- Lezione 3 (Input del ML): attributi *nominali* vs *numerici*, *istanze*
- Lezione 11 (Feature Engineering): one-hot encoding, imputazione dei mancanti
- Lezione 5 (Algoritmi di base): **1R** e **Naïve Bayes** (i due classificatori scelti)


## 0. Import e riproducibilità

Fissiamo il seme dei generatori casuali (come nei notebook del corso, es.
`np.random.seed(...)`): garantisce che i file estratti siano **sempre gli stessi**
a ogni esecuzione e che i risultati siano replicabili — requisito fondamentale per
un progetto valutato.


In [1]:
import pandas as pd
import numpy as np

In [2]:
SEED = 10
np.random.seed(SEED)

## 1. Caricamento

Il file usa il **punto e virgola** (`;`) come separatore, come gli altri CSV del
corso. Senza specificarlo, pandas leggerebbe tutto in un'unica colonna.

> Il percorso assume la struttura del repository: il dataset si trova in
> `../data/raw/`. Adattalo se necessario.


In [3]:
PATH = "../data/raw/bank-additional-full.csv"
df = pd.read_csv(PATH, sep=";")

In [4]:
print(f"Caricato dataset: {df.shape[0]} istanze x {df.shape[1]} attributi")

Caricato dataset: 41188 istanze x 21 attributi


In [5]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 2. Controllo qualità di base

### 2a. Duplicati
Le istanze identiche su tutti gli attributi non aggiungono informazione e
distorcono le frequenze (che Naïve Bayes usa per stimare le probabilità). Le rimuoviamo.


In [6]:
n_dup = df.duplicated().sum()
print(f"Istanze duplicate trovate: {n_dup}")

Istanze duplicate trovate: 12


In [7]:
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dataset dopo la rimozione: {df.shape[0]} istanze")

Dataset dopo la rimozione: 41176 istanze


### 2b. Distribuzione della classe
Verifichiamo lo **sbilanciamento** del target: ci serve consapevolezza per la
stratificazione (Lezione 8) che applicheremo nel Task 5.


In [8]:
df["y"].value_counts()

y
no     36537
yes     4639
Name: count, dtype: int64

In [9]:
perc = round(df["y"].value_counts(normalize=True)["yes"] * 100, 2)
print(f"Proporzione 'yes': {perc}%")

Proporzione 'yes': 11.27%


### 2c. Valori `unknown`
Nel dataset i valori mancanti sono codificati come la stringa `"unknown"` (lo
dichiara la documentazione). **Non sono celle vuote**, quindi pandas non li vede
come `NaN`: per questo UCI riporta "nessun valore mancante", anche se di fatto
mancano.

**Scelta:** per ora li manteniamo come categoria a sé. L'eventuale imputazione
(Lezione 11, `SimpleImputer`) sarà valutata nel Task 5 e applicata **solo** al
training set, per evitare *data leakage*.


In [10]:
nominali = df.select_dtypes(include=["object", "string"]).columns
for c in nominali:
    n = (df[c] == "unknown").sum()
    if n > 0:
        print(f"{c}: {n} unknown ({round(n/len(df)*100,1)}%)")

job: 330 unknown (0.8%)
marital: 80 unknown (0.2%)
education: 1730 unknown (4.2%)
default: 8596 unknown (20.9%)


housing: 990 unknown (2.4%)
loan: 990 unknown (2.4%)


## 3. Codifica della classe

Un classificatore matematico non interpreta le stringhe. Distinguiamo (Lezione 3):
- attributi **numerici** (continui): già pronti — `age`, `campaign`, `euribor3m`, …
- attributi **nominali** (categorici): da tradurre in numeri

Codifichiamo la **classe** `y` con `yes → 1`, `no → 0`. La classe `yes` (cliente che
sottoscrive il deposito) è la classe **positiva**, quella che vogliamo predire: per
convenzione vale 1.

> La codifica degli attributi nominali in ingresso (one-hot, Lezione 11) la faremo
> nel **Task 5** dentro una pipeline, perché dipende dal modello scelto e perché
> l'EDA del Task 3 è più leggibile sugli attributi nominali originali.


In [11]:
df["y"] = df["y"].map({"no": 0, "yes": 1})
print("Classe y codificata: no->0, yes->1 (yes = classe positiva)")

Classe y codificata: no->0, yes->1 (yes = classe positiva)


In [12]:
df["y"].value_counts()

y
0    36537
1     4639
Name: count, dtype: int64

## 4. Estrazione dei due file

### 4a. `training.csv`

**Scelta: usiamo tutto il dataset pulito** (~41.176 istanze). Motivazioni:
- la classe `yes` è rara: ogni istanza positiva è preziosa, sottocampionare
  ridurrebbe il segnale già scarso;
- il volume è gestibile per i modelli previsti;
- la separazione train/test vera e propria la faremo nel Task 5 con
  `train_test_split` (holdout stratificato, Lezione 8).

Salviamo il dataset pulito ma ancora leggibile (nominali come stringhe, classe già 0/1).


In [13]:
import os
os.makedirs("../data/processed", exist_ok=True)

In [14]:
training = df.copy()
training.to_csv("../data/processed/training.csv", index=False)
print(f"Salvato training.csv: {training.shape[0]} istanze x {training.shape[1]} attributi")

Salvato training.csv: 41176 istanze x 21 attributi


### 4b. `manuale.csv`

Per i calcoli **a mano** del Task 2 (due classificatori, gruppo da 2):
- **1R (1-Rule)** → costruisce una regola su **un solo attributo**, scegliendo
  quello che produce meno errori. Per gli attributi numerici (`age`, `campaign`)
  applica una **discretizzazione** in intervalli.
- **Naïve Bayes** → combina **tutti** gli attributi: i nominali tramite le frequenze
  (con stimatore di Laplace), i numerici (`age`, `campaign`) tramite la
  distribuzione gaussiana.

Manteniamo quindi un mix di attributi **nominali e numerici**: così possiamo
illustrare per entrambi i classificatori la gestione dei due tipi di attributo
(discretizzazione per 1R, gaussiana per Naïve Bayes).

Servono **poche istanze** (12, nel range 10–15 richiesto) e un mix **bilanciato**
yes/no: con soli `no` i calcoli sarebbero poco significativi.


In [15]:
feature_manuale = [
    "age", "campaign",                # numerici
    "job", "marital", "education",    # nominali
    "housing", "loan", "contact", "poutcome",
    "y",                              # classe
]

In [16]:
# 6 istanze 'yes' e 6 'no' -> set bilanciato (12 istanze)
yes_rows = df[df["y"] == 1].sample(n=6, random_state=SEED)
no_rows  = df[df["y"] == 0].sample(n=6, random_state=SEED)

In [17]:
manuale = pd.concat([yes_rows, no_rows])[feature_manuale]
# Mescoliamo l'ordine cosi' le classi non sono raggruppate
manuale = manuale.sample(frac=1, random_state=SEED).reset_index(drop=True)

In [18]:
manuale.to_csv("../data/processed/manuale.csv", index=False)
print(f"Salvato manuale.csv: {manuale.shape[0]} istanze x {manuale.shape[1]} attributi")

Salvato manuale.csv: 12 istanze x 10 attributi


In [19]:
manuale

,age,campaign,job,marital,education,housing,loan,contact,poutcome,y
0,35,3,admin.,single,professional.course,yes,no,cellular,nonexistent,1
1,53,1,blue-collar,married,unknown,no,no,cellular,nonexistent,0
2,36,3,blue-collar,married,basic.9y,yes,no,cellular,failure,0
3,61,1,retired,married,basic.4y,no,no,telephone,nonexistent,1
4,47,3,admin.,married,university.degree,no,no,telephone,nonexistent,0
5,36,4,blue-collar,married,unknown,yes,no,cellular,nonexistent,0
6,30,4,blue-collar,married,basic.6y,yes,no,cellular,nonexistent,0
7,31,1,admin.,single,high.school,yes,no,cellular,nonexistent,1
8,51,5,technician,married,university.degree,yes,no,cellular,nonexistent,1
9,35,1,blue-collar,divorced,basic.9y,no,no,cellular,failure,1


## Riepilogo

Abbiamo prodotto:
- **`training.csv`** — dataset pulito completo, pronto per EDA e addestramento
- **`manuale.csv`** — 12 istanze bilanciate con attributi adatti ai calcoli a mano

**Prossimo passo (Task 2):** definire a mano i due classificatori (Naïve Bayes e
KNN) su `manuale.csv`, illustrare i passi per adattarli ai dati, implementarli in
Python e valutarne le prestazioni.
